# ML Zoomcamp 2026 — Homework 2

## Machine Learning for Regression

This notebook contains my analysis and solutions for Homework 2 of the
DataTalksClub Machine Learning Zoomcamp 2026.

The goal is to prepare the dataset, build regression models, and evaluate
their performance using validation and test data.

### Dataset

`car_fuel_efficiency_2026.csv`

## 1. Setup and Data Loading

We start by importing the libraries required for data manipulation
and numerical calculations, then load the car fuel-efficiency dataset.

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = pd.read_csv(url)

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Question 1 — Missing Values

**Question:** Which column contains missing values?

Checking missing values is an important data-preparation step because missing observations may need to be handled before training a regression model.

In [2]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

**Answer:** `horsepower`

The `horsepower` column contains missing values, so it will need to be handled before modeling.

## Question 2 — Median Horsepower

**Question:** What is the median value of `horsepower`?

The median gives us the middle value of the horsepower distribution and
will also be useful when deciding how to handle missing values.

In [3]:
df['horsepower'].median()

254.0

**Answer:** `254`

The median horsepower is 254, which will be used later when comparing
different approaches for filling the missing values.

## Question 3 — Filling Missing Values

**Question:** Which approach gives better model performance when filling
the missing `horsepower` values?

We compare two approaches: filling the missing values with `0` and filling
them with the mean horsepower. The model performance will tell us which
approach works better for this dataset.

In [4]:
df = df[
    [
        'engine_displacement',
        'horsepower',
        'vehicle_weight',
        'model_year',
        'fuel_efficiency_mpg'
    ]
]

df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [5]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[n_train:n_train + n_val]
df_test = df.iloc[n_train + n_val:]

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [6]:
X_train = df_train.drop(columns=['fuel_efficiency_mpg']).copy()
X_val = df_val.drop(columns=['fuel_efficiency_mpg']).copy()

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

X_train.shape, X_val.shape

((6000, 4), (2000, 4))

In [7]:
X_train_zero = X_train.fillna(0)
X_val_zero = X_val.fillna(0)

X_train_zero.isnull().sum()

engine_displacement    0
horsepower             0
vehicle_weight         0
model_year             0
dtype: int64

### Q3 — Model with Missing Values Filled with 0

We first train a linear regression model after replacing missing
`horsepower` values with 0.

The model is trained on the training set and evaluated on the
validation set using RMSE.

In [8]:
def train_linear_regression(X, y):
    X = np.column_stack([np.ones(len(X)), X])
    
    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)
    
    w = XTX_inv @ X.T @ y
    
    return w

### Q3 — Validation RMSE

We use the trained weights to predict fuel efficiency on the validation set.
RMSE measures the typical size of the prediction error, with larger errors
having more influence on the result.

In [9]:
def predict(X, w):
    X = np.column_stack([np.ones(len(X)), X])
    return X @ w


def rmse(y, y_pred):
    error = y_pred - y
    return np.sqrt(np.mean(error ** 2))

In [10]:
w_zero = train_linear_regression(X_train_zero.values, y_train)

y_pred_zero = predict(X_val_zero.values, w_zero)

rmse_zero = rmse(y_val, y_pred_zero)

rmse_zero

2.181530423892005

**RMSE with `horsepower` filled with 0:** `2.1815`

### Q3 — Model with Missing Values Filled with the Mean

For the second approach, we replace the missing `horsepower` values with
the mean horsepower calculated from the training data.

Using the training data to calculate the mean avoids using information
from the validation set.

In [11]:
mean_hp = X_train['horsepower'].mean()

mean_hp

254.46338337605272

In [12]:
X_train_mean = X_train.fillna(mean_hp)
X_val_mean = X_val.fillna(mean_hp)

X_train_mean.isnull().sum()

engine_displacement    0
horsepower             0
vehicle_weight         0
model_year             0
dtype: int64

In [13]:
w_mean = train_linear_regression(X_train_mean.values, y_train)

y_pred_mean = predict(X_val_mean.values, w_mean)

rmse_mean = rmse(y_val, y_pred_mean)

rmse_mean

2.179390268643807

**Answer:** `With mean`

The model using the mean to fill missing `horsepower` values achieved a
slightly lower validation RMSE (2.17939) than the model using 0 (2.18153).

## Question 4 — Best Regularization

We now test regularized linear regression with different values of `r`.

Regularization adds a penalty to the model weights, which can help
control the size of the coefficients and improve validation performance.

For this question, missing `horsepower` values are filled with 0.

In [14]:
def train_linear_regression_reg(X, y, r):
    X = np.column_stack([np.ones(len(X)), X])
    
    XTX = X.T @ X
    XTX_reg = XTX + r * np.eye(XTX.shape[0])
    
    XTX_reg[0, 0] = XTX[0, 0]
    
    w = np.linalg.inv(XTX_reg) @ X.T @ y
    
    return w

In [15]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]

for r in r_values:
    w = train_linear_regression_reg(
        X_train_zero.values,
        y_train,
        r
    )
    
    y_pred = predict(X_val_zero.values, w)
    score = rmse(y_val, y_pred)
    
    print(r, round(score, 4))

0 2.1815
0.01 2.1815
0.1 2.1815
1 2.1815
5 2.1815
10 2.1815
100 2.1815


**Answer:** `0`

All tested regularization values produced the same validation RMSE when
rounded to four decimal places. Following the tie-breaking rule, the
smallest value of `r` is selected.

## Question 5 — Random Seed

We now repeat the train/validation/test split using different random seeds.

The purpose is to see how sensitive the validation RMSE is to the particular
random split of the data.

In [17]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

scores = []

for seed in seeds:
    np.random.seed(seed)

    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[n_train:n_train + n_val]

    X_train = df_train.drop(columns=['fuel_efficiency_mpg']).copy()
    X_val = df_val.drop(columns=['fuel_efficiency_mpg']).copy()

    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values

    X_train = X_train.fillna(0)
    X_val = X_val.fillna(0)

    w = train_linear_regression(X_train.values, y_train)

    y_pred = predict(X_val.values, w)

    score = rmse(y_val, y_pred)
    scores.append(score)

    print(seed, round(score, 4))

np.std(scores)

0 2.1815
1 2.1781
2 2.1796
3 2.1817
4 2.1796
5 2.1815
6 2.1812
7 2.1783
8 2.1794
9 2.1817


0.0013503039475245265

In [18]:
round(np.std(scores), 3)

0.001

## Question 6 — Evaluation on Test Set

For the final evaluation, we use seed 9, combine the training and
validation datasets, fill missing `horsepower` values with 0, and train
a regularized linear regression model with `r = 0.001`.

The final RMSE is calculated on the test dataset.

In [19]:
np.random.seed(9)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[n_train:n_train + n_val]
df_test = df.iloc[n_train + n_val:]

df_full_train = pd.concat([df_train, df_val])

X_full_train = df_full_train.drop(columns=['fuel_efficiency_mpg']).copy()
y_full_train = df_full_train['fuel_efficiency_mpg'].values

X_test = df_test.drop(columns=['fuel_efficiency_mpg']).copy()
y_test = df_test['fuel_efficiency_mpg'].values

X_full_train = X_full_train.fillna(0)
X_test = X_test.fillna(0)

X_full_train.shape, X_test.shape

((8000, 4), (2000, 4))

In [20]:
r = 0.001

w = train_linear_regression_reg(
    X_full_train.values,
    y_full_train,
    r
)

y_pred = predict(X_test.values, w)

rmse_test = rmse(y_test, y_pred)

rmse_test

2.2775888434920546

In [21]:
df.shape, df['horsepower'].isna().sum(), df['fuel_efficiency_mpg'].mean()

((10000, 5), 877, 29.9977)

In [22]:
df = pd.read_csv(url)

df = df[
    [
        'engine_displacement',
        'horsepower',
        'vehicle_weight',
        'model_year',
        'fuel_efficiency_mpg'
    ]
].copy()

df.shape

(10000, 5)

In [23]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [24]:
scores = []

for seed in range(10):
    np.random.seed(seed)

    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]

    X_train = df_train.drop(columns=['fuel_efficiency_mpg']).copy()
    X_val = df_val.drop(columns=['fuel_efficiency_mpg']).copy()

    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values

    X_train = X_train.fillna(0)
    X_val = X_val.fillna(0)

    w = train_linear_regression(X_train.values, y_train)

    y_pred = predict(X_val.values, w)

    score = rmse(y_val, y_pred)
    scores.append(score)

np.std(scores), round(np.std(scores), 3)

(0.028781274078187532, 0.029)

**Answer:** `0.029`

The standard deviation of the validation RMSE values across the 10 random
seeds is approximately 0.029.

## Question 6 — Final Test RMSE

For the final evaluation, we use seed 9, combine the training and
validation datasets, fill missing `horsepower` values with 0, and train
a regularized linear regression model with `r = 0.001`.

The final model is evaluated on the test set using RMSE.

In [25]:
np.random.seed(9)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

df_full_train = pd.concat([df_train, df_val])

X_full_train = df_full_train.drop(columns=['fuel_efficiency_mpg']).copy()
y_full_train = df_full_train['fuel_efficiency_mpg'].values

X_test = df_test.drop(columns=['fuel_efficiency_mpg']).copy()
y_test = df_test['fuel_efficiency_mpg'].values

X_full_train = X_full_train.fillna(0)
X_test = X_test.fillna(0)

X_full_train.shape, X_test.shape

((8000, 4), (2000, 4))

In [26]:
r = 0.001

w = train_linear_regression_reg(
    X_full_train.values,
    y_full_train,
    r
)

y_pred = predict(X_test.values, w)

rmse_test = rmse(y_test, y_pred)

rmse_test

2.235774702816487

**Answer:** `2.236`

The final model achieves a test RMSE of approximately 2.236.

---

## Final Notes and What I Learned

This homework was mainly about understanding the basic workflow of a
regression problem rather than just getting the answers to the questions.

I started by checking the dataset and identifying missing values. The
`horsepower` column had missing values, and its median was 254. I then
worked through different ways of handling those missing values and saw
how the choice of preprocessing can affect model performance.

For Question 3, I compared two approaches for filling the missing
`horsepower` values: replacing them with `0` and replacing them with the
mean calculated from the training data. The mean-based approach gave a
slightly lower validation RMSE, so the answer was **With mean**.

For Question 4, I tested different regularization values:

`0, 0.01, 0.1, 1, 5, 10, 100`

The RMSE values were the same after rounding to four decimal places.
Because the homework specifies choosing the smallest value when there
is a tie, the answer was **0**.

### Why I rechecked Questions 5 and 6

There was an important debugging/learning moment while working on this
homework.

My first calculation for Question 5 produced a standard deviation of
approximately `0.001`, which did not match any of the available MCQ
options. Instead of simply choosing the closest option, I stopped and
checked the calculation.

The problem was not the dataset. I had been reusing and modifying
variables such as `df`, `df_train`, `df_val`, and other intermediate
objects while working through the earlier questions. This meant that
the later calculations were not being reproduced from a completely
clean state.

Rather than forcing an answer to match the MCQ, I restarted the
relevant calculations from a fresh copy of the original dataset and
followed the required procedure again.

For Question 5, I used the random seeds from `0` through `9`, filled the
missing `horsepower` values with `0`, calculated the validation RMSE for
each split, and then calculated the standard deviation of those RMSE
values.

The result was:

`0.028781...`

which rounds to **0.029**.

I followed the same approach for Question 6 instead of trusting the
earlier result. Using seed `9`, I combined the training and validation
sets, filled missing `horsepower` values with `0`, trained the
regularized linear regression model with `r = 0.001`, and evaluated it
on the test set.

The final RMSE was:

`2.235774...`

which rounds to **2.236**.

This was a useful reminder that in machine learning, getting a number
that looks reasonable is not enough. If the result does not agree with
the expected setup or available choices, it is better to stop, inspect
the process, and reproduce the calculation from a clean state.

### Final Answers

- **Q1:** `horsepower`
- **Q2:** `254`
- **Q3:** `With mean`
- **Q4:** `0`
- **Q5:** `0.029`
- **Q6:** `2.236`

### Main Takeaways

- Missing values need to be handled before training a model.
- The way missing values are filled can affect model performance.
- Validation RMSE can change depending on the random train/validation
  split.
- Regularization controls the size of model coefficients.
- A final model should be evaluated on data that was not used for
  training.
- Reproducibility matters: random seeds and a clean data-processing
  pipeline are important when comparing results.
- When a result does not match the expected setup, I should investigate
  the calculation instead of simply selecting the closest answer.

---

## Notes

This notebook represents my own work and learning process as part of
Machine Learning Zoomcamp 2026.

The analysis, calculations, and explanations were completed for
educational purposes and reflect my understanding of the concepts
covered in this exercise.

Please do not submit, reproduce, or present this work as your own.
If you use any part of this notebook for learning or reference,
please provide appropriate attribution.

**Author:** Shubham Kulkarni  
**Program:** Machine Learning Zoomcamp 2026